In [1]:
from astropy.table import join,Table,Column,vstack
import numpy as np
fpath='./data/sersic_cat.fits'
dat=Table.read(fpath) 
dat.info('stats')

FileNotFoundError: [Errno 2] No such file or directory: './data/sersic_cat.fits'

In [2]:
import catalogue_analysis as ca
import colour_lookup as col
import numpy as np
from astropy.table import join,Table,Column,vstack
#Specify regions to loop over
regions=('S','N')

#Specify data release  Y1/Y3

Y3=True

dat.remove_columns(['zmin','zmax','vmin','vmax','v'])
datfull=dat

#Random sample and apply all cuts from the start for a quick run through
Sel=ca.selection('N')# import North selection cuts  
smaskN= (datfull['Z'] > Sel['zmin']) & (datfull['Z'] < Sel['zmax']) & (np.random.rand(datfull['Z'].size)<Sel['f_ran'])\
& (datfull['rmag'] < Sel['faint'])  & (datfull['rmag'] > Sel['bright']) & (datfull['reg']=='N') 
Sel=ca.selection('S') # import South selection cuts    
smaskS= (datfull['Z'] > Sel['zmin']) & (datfull['Z'] < Sel['zmax']) &(np.random.rand(datfull['Z'].size)<Sel['f_ran']) \
& (datfull['rmag'] < Sel['faint'])  & (datfull['rmag'] > Sel['bright']) & (datfull['reg']=='S')
smask= np.logical_or(smaskN,smaskS) #True for objects making cuts in North or South
dat=datfull[smask]    
print('South sample size:',smaskS.sum())
print('North sample size:',smaskN.sum())
dat.info('stats') 
    




ModuleNotFoundError: No module named 'colour_lookup'

In [ ]:
#sky plot
#ca.sky_plot(dat,regions)
import matplotlib.pyplot as plt
plt.scatter(dat['Z'],dat['rmag'], marker='.' , linewidths=0,s=0.25,alpha=0.2)
plt.ylim([12,20])
plt.xlim([0.0,0.6])
plt.xlabel('$z$')
plt.ylabel('$r$')
plt.show()

In [ ]:
# Compute rest frame colours and absolute magnitudes using precomputed k-corrections and the Evolution parameter specified in catalogue_analysis.selection()
# If fresh=True then the k-ccorections are recomputed using John Moustakas's Fast Spec Catalogue which is used to fit k-corrections
fresh=False
if fresh: 
    fsf=ca.read_fsf('data/fastspec-iron-main-bright.fits')
else :
    fsf='null'
    
#compute rest frame colours and absolute magnitudes using k-corrections computed from the fsf catalogue
dat.info('stats')
ca.recompute_rest_col_mag(dat,regions,fsf,fresh=fresh,test_plots=False)
dat.info('stats')

In [ ]:
# Make a few basic QA plots

#Plot k-corrections to check they are well behaved
ca.plot_kcorr(regions)
#colour-magnitude distributions
ca.plot_col_mag(dat,regions)
#sky plot
ca.sky_plot(dat,regions)

In [ ]:
# Compute zmax and vmax values
ca.compute_zmax_vmax(dat,regions)

#Plots of how zmax depends on absolute magnitude, colour and redshift
ca.plot_zmax_absmag(dat)
ca.plot_zmax_z(dat)
#Plots of how zmin depends on absolute magnitude, colour and redshift
ca.plot_zmin_absmag(dat)
ca.plot_zmin_z(dat)

In [ ]:
#colour-magnitude distributions
ca.plot_col_mag(dat,regions)
#ca.plot_col_mag_withvmax(dat,regions)

In [ ]:
#redshift distribution
ca.hist_nz(dat,regions)

In [ ]:
#Plot overall V/Vmax distribution
ca.plot_v_vmax(dat,regions)

In [ ]:
#1/Vmax LF estimate
log_phi_vmax,log_phi_low,log_phi_hi,magbins=ca.lumfun_vmax(dat,regions,ratio=False)

In [ ]:
# Petrosian Radii and Flux Ratios
#
import numpy as np
from scipy.special import gammainc, gamma
from scipy.optimize import fsolve
# Solves for the parameter xp that is related to the Petrosian radius as described in
# https://www.overleaf.com/read/npcktptmbqtf#747974 
#
# Define the function to solve
def equation_to_solve(x, n):
    # Ensure x is positive to avoid undefined behaviour
    if x <= 0:
        return np.inf
    # The integral is the lower incomplete gamma function
    integral = gamma(2 * n) * gammainc(2 * n, x)  # gammainc gives the normalized incomplete gamma
    lhs = 0.2* 2.0* n * (x ** (-2 * n)) * np.exp(x) * integral  # LHS of the equation
    # The 0.2 above is the choice made by SDSS (see Blanton et al 2001)
    return lhs - 1.0  # The equation we want to solve


# Function to evaluate the ratio Petrosian to total flux as described in
# https://www.overleaf.com/read/npcktptmbqtf#747974 
#
def flux_ratio(xmax, n):
    # Numerator: lower incomplete Gamma function
    ratio = gammainc(2 * n, xmax)
    return ratio

x0=4.0 #initial guess for xp
ntab=np.linspace(0.2, 6.0, num=60)
delta_tab=np.zeros(ntab.size)
for i in range(0,ntab.size):
    n=ntab[i]
    xp=fsolve(equation_to_solve, x0, args=(n,))
    rp=xp**n    # relationship between xp and the Petrosian radius rp
    rmax=2.0*rp # The factor 2.0 is choice made by SDSS (see Blanton et al 2001)
    xmax=rmax**(1/n) # Convert back to x for which the flux ratio is a standard function
    x0=xp  #update guess
    fratio = flux_ratio(xmax, n)  # Petrosian flux ratio
    delta_tab[i]=-2.5*np.log10(fratio[0])
    
print(ntab)
print(delta_tab)
               
print("n_sersic      flux_ratio   Delta_Mag")
# Loop over values of Sersic index n
for i in range(1, 60):
    n=0.2 + 0.1*(i-1)

    # Initial guess for xp
    x0 = 4.0  # Starting point for numerical solver
    # Solve using fsolve to find xp
    xp = fsolve(equation_to_solve, x0, args=(n,))
    rp=xp**n    # relationship between xp and the Petrosian radius rp
    rmax=2.0*rp # The factor 2.0 is choice made by SDSS (see Blanton et al 2001)
    xmax=rmax**(1/n) # Convert back to x for which the flux ratio is a standard function

    fratio = flux_ratio(xmax, n)  # Petrosian flux ratio
    print(f"{n:.1f}      &         {fratio[0]:.3f}   &  {-2.5*np.log10(fratio[0]):.3f} \\\\")

    #print(f"n:{n} xp:{xp} fratio:{fratio} at root:{equation_to_solve(xp, n)}")



In [ ]:
dat['rmag']=dat['rmag']+np.interp(dat['SERSIC'],ntab,delta_tab)
dat['ABSMAG_RP1']=dat['rmag']+np.interp(dat['SERSIC'],ntab,delta_tab)

In [ ]:

dat.remove_columns(['zmin','zmax','vmin','vmax','v'])
datfull=dat

#Random sample and apply all cuts from the start for a quick run through
Sel=ca.selection('N')# import North selection cuts  
smaskN= (datfull['Z'] > Sel['zmin']) & (datfull['Z'] < Sel['zmax']) & (np.random.rand(datfull['Z'].size)<Sel['f_ran'])\
& (datfull['rmag'] < Sel['faint'])  & (datfull['rmag'] > Sel['bright']) & (datfull['reg']=='N') 
Sel=ca.selection('S') # import South selection cuts    
smaskS= (datfull['Z'] > Sel['zmin']) & (datfull['Z'] < Sel['zmax']) &(np.random.rand(datfull['Z'].size)<Sel['f_ran']) \
& (datfull['rmag'] < Sel['faint'])  & (datfull['rmag'] > Sel['bright']) & (datfull['reg']=='S')
smask= np.logical_or(smaskN,smaskS) #True for objects making cuts in North or South
dat=datfull[smask]    
print('South sample size:',smaskS.sum())
print('North sample size:',smaskN.sum())
dat.info('stats') 
    




In [ ]:
#sky plot
#ca.sky_plot(dat,regions)
import matplotlib.pyplot as plt
plt.scatter(dat['Z'],dat['rmag'], marker='.' , linewidths=0,s=0.25,alpha=0.2)
plt.ylim([12,20])
plt.xlim([0.0,0.6])
plt.xlabel('$z$')
plt.ylabel('$r$')
plt.show()

In [ ]:
# Compute rest frame colours and absolute magnitudes using precomputed k-corrections and the Evolution parameter specified in catalogue_analysis.selection()
# If fresh=True then the k-ccorections are recomputed using John Moustakas's Fast Spec Catalogue which is used to fit k-corrections
fresh=False
if fresh: 
    fsf=ca.read_fsf('data/fastspec-iron-main-bright.fits')
else :
    fsf='null'
    
#compute rest frame colours and absolute magnitudes using k-corrections computed from the fsf catalogue
dat.info('stats')
ca.recompute_rest_col_mag(dat,regions,fsf,fresh=fresh,test_plots=False)
dat.info('stats')

In [ ]:
# Make a few basic QA plots

#Plot k-corrections to check they are well behaved
ca.plot_kcorr(regions)
#colour-magnitude distributions
ca.plot_col_mag(dat,regions)
#sky plot
ca.sky_plot(dat,regions)

In [ ]:
# Compute zmax and vmax values
ca.compute_zmax_vmax(dat,regions)

#Plots of how zmax depends on absolute magnitude, colour and redshift
ca.plot_zmax_absmag(dat)
ca.plot_zmax_z(dat)
#Plots of how zmin depends on absolute magnitude, colour and redshift
ca.plot_zmin_absmag(dat)
ca.plot_zmin_z(dat)

In [ ]:
#colour-magnitude distributions
ca.plot_col_mag(dat,regions)
#ca.plot_col_mag_withvmax(dat,regions)

In [ ]:
#colour-magnitude distributions
ca.plot_col_mag(dat,regions)
#ca.plot_col_mag_withvmax(dat,regions)

In [ ]:
#redshift distribution
ca.hist_nz(dat,regions)

In [ ]:
#Plot overall V/Vmax distribution
ca.plot_v_vmax(dat,regions)

In [ ]:
#1/Vmax LF estimate
log_phi_vmax,log_phi_low,log_phi_hi,magbins=ca.lumfun_vmax(dat,regions,ratio=False)